# Fingerprint Enhancement — Full Pipeline with Skeletonization
**Project:** Signal & Image Processing Mini-Project  
**Method:** Gabor Filtering → Background Masking → Binarization → Skeletonization  
**Dataset:** FVC2000 DB1_B (`.tif` fingerprint images)  

This notebook is the **complete pipeline**, producing a 1-pixel-wide skeleton ready for minutiae (ridge-ending / bifurcation) extraction.

## Stage 1 — Import Libraries & Load Image

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

# ── Path: place fingerprints folder next to this notebook ──────────────────
IMAGE_PATH = os.path.join("fingerprints", "fingerprints", "DB1_B", "109_6.tif")
gray = cv.imread(IMAGE_PATH, cv.IMREAD_GRAYSCALE)

if gray is None:
    raise FileNotFoundError(
        f"Image not found at '{IMAGE_PATH}'. "
        "Make sure 'fingerprints/DB1_B/' is in the same folder as this notebook."
    )

print(f"Image loaded. Shape: {gray.shape}, dtype: {gray.dtype}")
plt.figure(figsize=(4, 4))
plt.imshow(gray, cmap='gray')
plt.title("Original Fingerprint")
plt.axis('off')
plt.tight_layout()
plt.show()


## Stage 2 — Noise Removal, Contrast Enhancement & Normalisation

In [ ]:
# ── Noise removal ──────────────────────────────────────────────────────────
median_filtered   = cv.medianBlur(gray, 3)
gaussian_filtered = cv.GaussianBlur(median_filtered, (3, 3), 0)

# ── Contrast enhancement ───────────────────────────────────────────────────
equalized = cv.equalizeHist(gaussian_filtered)
gamma_val = 0.5
power_law = np.uint8(np.power(equalized / 255.0, gamma_val) * 255)

# ── Intensity normalisation ────────────────────────────────────────────────
smqt_stage = cv.normalize(power_law, None, 0, 255, cv.NORM_MINMAX)

plt.figure(figsize=(14, 3))
for i, (img, title) in enumerate([(gray, "1. Original"),
                                   (gaussian_filtered, "2. Noise Removed"),
                                   (power_law, "3. Gamma Corrected"),
                                   (smqt_stage, "4. Normalised")], 1):
    plt.subplot(1, 4, i)
    plt.imshow(img, cmap='gray')
    plt.title(title, fontsize=9)
    plt.axis('off')
plt.suptitle("Stage 2: Preprocessing", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Stage 3 — Background Mask Generation
**Otsu's thresholding** automatically finds the optimal threshold to separate the fingerprint region from the background.  
A large closing kernel (25×25) fills holes and creates a clean binary mask.

In [ ]:
# Otsu's method: automatically computes the best threshold
# THRESH_BINARY_INV makes the fingerprint area WHITE (255) and background BLACK (0)
_, mask = cv.threshold(smqt_stage, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

# Morphological closing with a large kernel fills gaps inside the fingerprint region
mask_kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (25, 25))
mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, mask_kernel)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(smqt_stage, cmap='gray')
plt.title("Normalised Input")
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(mask, cmap='gray')
plt.title("Background Mask (Otsu + Closing)")
plt.axis('off')
plt.suptitle("Stage 3: Background Mask Generation", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Stage 4 — Gabor Filter Bank (8 Orientations)
Gabor filters are band-pass filters tuned to specific ridge orientations and spatial frequencies.

In [ ]:
# ── Gabor parameters ───────────────────────────────────────────────────────
ksize   = 15    # kernel size in pixels
sigma   = 1.0   # Gaussian envelope std dev
lamda   = 0.5   # sinusoidal wavelength
gamma_g = 0.5   # spatial aspect ratio
# Note: phi omitted here → defaults to 0 (cosine phase), which is standard

# 8 orientations: 0°, 22.5°, 45°, 67.5°, 90°, 112.5°, 135°, 157.5°
thetas = np.arange(0, np.pi, np.pi / 8)

gabor_outputs = []
plt.figure(figsize=(16, 8))

for i, theta in enumerate(thetas):
    # Create Gabor kernel for orientation theta
    gabor_kernel = cv.getGaborKernel(
        (ksize, ksize), sigma, theta, lamda, gamma_g, ktype=cv.CV_32F
    )
    # Convolve the normalised image with this kernel
    gabor_response = cv.filter2D(smqt_stage, cv.CV_8U, gabor_kernel)
    gabor_outputs.append(gabor_response)

    plt.subplot(2, 4, i + 1)
    plt.imshow(gabor_response, cmap='gray')
    plt.title(f"θ = {np.degrees(theta):.1f}°", fontsize=10)
    plt.axis('off')

plt.suptitle("Stage 4: Gabor Filter Bank — 8 Orientations", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Stage 5 — Max-Fusion, Masking & Binarization
1. **Max-fusion**: pixel-wise maximum across all 8 responses  
2. **Mask application**: removes Gabor noise from the background  
3. **Otsu binarization**: converts to clean black-and-white

In [ ]:
# ── Step 5a: Combine the two dominant diagonal angles ─────────────────────
# (22.5° and 67.5° capture most ridge orientations in typical fingerprints)
res_22 = gabor_outputs[1]   # 22.5°
res_67 = gabor_outputs[3]   # 67.5°
combined = cv.addWeighted(res_22, 1.0, res_67, 1.0, 0)

# ── Step 5b: Morphological closing — joins broken ridge segments ───────────
kernel_close = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))
joined = cv.morphologyEx(combined, cv.MORPH_CLOSE, kernel_close)

# ── Step 5c: Apply background mask — zeros out non-fingerprint pixels ──────
# bitwise_and: only keeps pixels where mask == 255
masked_final = cv.bitwise_and(joined, joined, mask=mask)

# ── Step 5d: Otsu binarization — produces clean black-and-white template ───
_, final_bw = cv.threshold(masked_final, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
# Final median blur to remove isolated noise specks before skeletonization
final_bw = cv.medianBlur(final_bw, 3)

plt.figure(figsize=(16, 4))
for i, (img, title) in enumerate([(combined,      "5a. Combined (22.5°+67.5°)"),
                                   (joined,        "5b. After Closing"),
                                   (masked_final,  "5c. After Masking"),
                                   (final_bw,      "5d. Binarized (Otsu)")], 1):
    plt.subplot(1, 4, i)
    plt.imshow(img, cmap='gray')
    plt.title(title, fontsize=9)
    plt.axis('off')
plt.suptitle("Stage 5: Fusion → Masking → Binarization", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Stage 6 — Skeletonization (Zhang-Suen Thinning)
Skeletonization reduces each ridge to a **1-pixel-wide centreline**.  
This is a prerequisite for minutiae extraction — ridge endings and bifurcations can only be reliably detected on a thinned skeleton.  

The algorithm used is iterative morphological thinning (Zhang-Suen approach):  
repeatedly erode the image and subtract what was lost, accumulating the skeleton.

In [ ]:
# ── Zhang-Suen style iterative thinning via morphological operations ───────
skeleton  = np.zeros(final_bw.shape, np.uint8)  # accumulator for skeleton pixels
temp_img  = final_bw.copy()
element   = cv.getStructuringElement(cv.MORPH_CROSS, (3, 3))

iteration = 0
while True:
    eroded   = cv.erode(temp_img, element)           # shrink by one pixel layer
    temp     = cv.dilate(eroded, element)             # re-dilate to see what was removed
    temp     = cv.subtract(temp_img, temp)            # the "outer ring" pixels
    skeleton = cv.bitwise_or(skeleton, temp)          # add ring to skeleton
    temp_img = eroded.copy()                          # continue from eroded image
    iteration += 1
    if cv.countNonZero(temp_img) == 0:               # stop when nothing left to erode
        break

print(f"Skeletonization completed in {iteration} iterations.")
print(f"Non-zero skeleton pixels: {cv.countNonZero(skeleton)}")


## Stage 7 — Final Display & Comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(final_bw, cmap='gray')
plt.title("Binary Fingerprint Template", fontsize=11)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(skeleton, cmap='gray')
plt.title("Skeletonized — Minutiae Ready", fontsize=11)
plt.axis('off')

plt.suptitle("Stage 6–7: Skeletonization Result", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Full Pipeline Summary

In [ ]:
stages = [
    (gray,         "1. Original"),
    (smqt_stage,   "2. Preprocessed"),
    (mask,         "3. Background Mask"),
    (masked_final, "4. Gabor + Masked"),
    (final_bw,     "5. Binary Template"),
    (skeleton,     "6. Skeleton"),
]

plt.figure(figsize=(20, 4))
for i, (img, title) in enumerate(stages, 1):
    plt.subplot(1, 6, i)
    plt.imshow(img, cmap='gray')
    plt.title(title, fontsize=9)
    plt.axis('off')
plt.suptitle("Complete Fingerprint Enhancement Pipeline", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Save final outputs
cv.imwrite("skeleton_output.tif", skeleton)
cv.imwrite("binary_template.tif", final_bw)
print("Outputs saved: 'skeleton_output.tif' and 'binary_template.tif'")
